# Gaussian Filtering Advanced - Result Visualization

**Purpose**: Visualize and analyze results from gaussian_filtering_advanced experiment

**Dataset**: Results from `/content/drive/MyDrive/model1/runs/gaussian_filtering_advanced/`

**Analysis**:
1. Confidence threshold optimization
2. Stage1 vs Stage2 threshold heatmaps
3. Precision-Recall tradeoffs
4. Best configuration identification
5. PCA analysis of results

## 1. Setup: Mount Drive & Install Dependencies

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required packages
!pip install matplotlib seaborn pandas numpy scikit-learn -q

In [ ]:
# Imports
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import os

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## 2. Load Results Data

In [ ]:
# Path to results (update if needed)
results_path = '/content/drive/MyDrive/model1/runs/gaussian_filtering_advanced/all_results.json'

# Load JSON data
with open(results_path, 'r') as f:
    results = json.load(f)

print(f"Loaded {len(results)} result configurations")
print(f"\nFirst result sample:")
print(json.dumps(results[0], indent=2))

In [ ]:
# Convert to pandas DataFrame for easier analysis
df = pd.DataFrame(results)

# Extract confusion matrix values
df['TN'] = df['confusion_matrix'].apply(lambda x: x[0][0])
df['FP'] = df['confusion_matrix'].apply(lambda x: x[0][1])
df['FN'] = df['confusion_matrix'].apply(lambda x: x[1][0])
df['TP'] = df['confusion_matrix'].apply(lambda x: x[1][1])

print("\nDataFrame shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nSummary statistics:")
print(df[['accuracy', 'precision', 'recall', 'f1', 'specificity', 'stage1_pass_rate']].describe())

## 3. Find Best Configurations

In [ ]:
# Find best configurations by different metrics
best_configs = {
    'accuracy': df.loc[df['accuracy'].idxmax()],
    'precision': df.loc[df['precision'].idxmax()],
    'recall': df.loc[df['recall'].idxmax()],
    'f1': df.loc[df['f1'].idxmax()],
    'specificity': df.loc[df['specificity'].idxmax()]
}

print("Best Configurations by Metric:")
print("=" * 80)
for metric, config in best_configs.items():
    print(f"\n{metric.upper()}:")
    print(f"  Stage1 Confidence: {config['confidence_stage1']:.2f}")
    print(f"  Stage2 Confidence: {config['confidence_stage2']:.2f}")
    print(f"  Accuracy: {config['accuracy']:.4f}")
    print(f"  Precision: {config['precision']:.4f}")
    print(f"  Recall: {config['recall']:.4f}")
    print(f"  F1: {config['f1']:.4f}")
    print(f"  Specificity: {config['specificity']:.4f}")
    print(f"  Stage1 Pass Rate: {config['stage1_pass_rate']:.4f}")

## 4. Heatmaps: Stage1 vs Stage2 Confidence Thresholds

In [ ]:
# Create pivot tables for heatmaps
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1']

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, metric in enumerate(metrics_to_plot):
    pivot = df.pivot_table(
        values=metric,
        index='confidence_stage1',
        columns='confidence_stage2',
        aggfunc='mean'
    )
    
    sns.heatmap(
        pivot,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        ax=axes[idx],
        cbar_kws={'label': metric.capitalize()}
    )
    
    axes[idx].set_title(f'{metric.capitalize()} vs Confidence Thresholds', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Stage2 Confidence Threshold', fontsize=12)
    axes[idx].set_ylabel('Stage1 Confidence Threshold', fontsize=12)

plt.tight_layout()
plt.savefig('/content/gaussian_filtering_advanced_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: /content/gaussian_filtering_advanced_heatmaps.png")

## 5. Performance Metrics Comparison

In [ ]:
# Plot performance metrics across different Stage1 confidence thresholds
stage1_thresholds = sorted(df['confidence_stage1'].unique())

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

metrics = ['accuracy', 'precision', 'recall', 'f1']
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

for idx, (metric, color) in enumerate(zip(metrics, colors)):
    for stage1_conf in stage1_thresholds:
        subset = df[df['confidence_stage1'] == stage1_conf]
        axes[idx].plot(
            subset['confidence_stage2'],
            subset[metric],
            marker='o',
            label=f'Stage1={stage1_conf:.2f}',
            alpha=0.7
        )
    
    axes[idx].set_xlabel('Stage2 Confidence Threshold', fontsize=12)
    axes[idx].set_ylabel(metric.capitalize(), fontsize=12)
    axes[idx].set_title(f'{metric.capitalize()} vs Stage2 Confidence', fontsize=14, fontweight='bold')
    axes[idx].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/gaussian_filtering_advanced_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: /content/gaussian_filtering_advanced_metrics.png")

## 6. Precision-Recall Tradeoff

In [ ]:
# Precision-Recall curve for different Stage1 thresholds
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

for stage1_conf in stage1_thresholds:
    subset = df[df['confidence_stage1'] == stage1_conf]
    ax.scatter(
        subset['recall'],
        subset['precision'],
        label=f'Stage1={stage1_conf:.2f}',
        s=100,
        alpha=0.6
    )

# Mark best F1 point
best_f1 = best_configs['f1']
ax.scatter(
    [best_f1['recall']],
    [best_f1['precision']],
    color='red',
    s=300,
    marker='*',
    edgecolors='black',
    linewidths=2,
    label=f'Best F1={best_f1["f1"]:.4f}',
    zorder=10
)

ax.set_xlabel('Recall', fontsize=14)
ax.set_ylabel('Precision', fontsize=14)
ax.set_title('Precision-Recall Tradeoff', fontsize=16, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

# Add diagonal line (F1 iso-lines)
for f1_val in [0.5, 0.6, 0.7, 0.8, 0.9]:
    recall_range = np.linspace(0.01, 1, 100)
    precision_line = (f1_val * recall_range) / (2 * recall_range - f1_val)
    precision_line = np.clip(precision_line, 0, 1)
    ax.plot(recall_range, precision_line, '--', color='gray', alpha=0.3, linewidth=0.8)
    ax.text(0.95, precision_line[-1], f'F1={f1_val}', fontsize=8, alpha=0.5)

plt.tight_layout()
plt.savefig('/content/gaussian_filtering_advanced_pr_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: /content/gaussian_filtering_advanced_pr_curve.png")

## 7. Stage1 Pass Rate Analysis

In [ ]:
# Analyze Stage1 pass rate impact
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pass rate vs Accuracy
for stage1_conf in stage1_thresholds:
    subset = df[df['confidence_stage1'] == stage1_conf]
    axes[0].scatter(
        subset['stage1_pass_rate'],
        subset['accuracy'],
        label=f'Stage1={stage1_conf:.2f}',
        s=100,
        alpha=0.6
    )

axes[0].set_xlabel('Stage1 Pass Rate', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Accuracy vs Stage1 Pass Rate', fontsize=14, fontweight='bold')
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0].grid(True, alpha=0.3)

# Pass rate vs F1
for stage1_conf in stage1_thresholds:
    subset = df[df['confidence_stage1'] == stage1_conf]
    axes[1].scatter(
        subset['stage1_pass_rate'],
        subset['f1'],
        label=f'Stage1={stage1_conf:.2f}',
        s=100,
        alpha=0.6
    )

axes[1].set_xlabel('Stage1 Pass Rate', fontsize=12)
axes[1].set_ylabel('F1 Score', fontsize=12)
axes[1].set_title('F1 Score vs Stage1 Pass Rate', fontsize=14, fontweight='bold')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/gaussian_filtering_advanced_pass_rate.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: /content/gaussian_filtering_advanced_pass_rate.png")

## 8. PCA Analysis

In [ ]:
# PCA on performance metrics
features = ['accuracy', 'precision', 'recall', 'f1', 'specificity', 'stage1_pass_rate']
X = df[features].values

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Plot PCA
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Colored by Stage1 confidence
scatter1 = axes[0].scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df['confidence_stage1'],
    cmap='viridis',
    s=100,
    alpha=0.6
)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
axes[0].set_title('PCA: Colored by Stage1 Confidence', fontsize=14, fontweight='bold')
plt.colorbar(scatter1, ax=axes[0], label='Stage1 Confidence')
axes[0].grid(True, alpha=0.3)

# Colored by F1 score
scatter2 = axes[1].scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df['f1'],
    cmap='RdYlGn',
    s=100,
    alpha=0.6
)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
axes[1].set_title('PCA: Colored by F1 Score', fontsize=14, fontweight='bold')
plt.colorbar(scatter2, ax=axes[1], label='F1 Score')
axes[1].grid(True, alpha=0.3)

# Mark best F1 point
best_f1_idx = df['f1'].idxmax()
axes[1].scatter(
    [X_pca[best_f1_idx, 0]],
    [X_pca[best_f1_idx, 1]],
    color='red',
    s=300,
    marker='*',
    edgecolors='black',
    linewidths=2,
    zorder=10
)

plt.tight_layout()
plt.savefig('/content/gaussian_filtering_advanced_pca.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: /content/gaussian_filtering_advanced_pca.png")
print(f"\nPCA Explained Variance:")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.2%}")
print(f"  Total: {pca.explained_variance_ratio_.sum():.2%}")

## 9. Top Configurations Summary

In [ ]:
# Get top 10 configurations by F1 score
top_configs = df.nlargest(10, 'f1')[[
    'confidence_stage1', 'confidence_stage2', 
    'accuracy', 'precision', 'recall', 'f1', 'specificity', 'stage1_pass_rate'
]].reset_index(drop=True)

print("Top 10 Configurations by F1 Score:")
print("=" * 100)
print(top_configs.to_string(index=True))

# Visualize top 10
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

x = np.arange(len(top_configs))
width = 0.15

metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'specificity']
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']

for idx, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
    offset = width * (idx - 2)
    ax.bar(x + offset, top_configs[metric], width, label=metric.capitalize(), color=color, alpha=0.8)

ax.set_xlabel('Configuration Rank', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Top 10 Configurations Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f"#{i+1}" for i in range(len(top_configs))])
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('/content/gaussian_filtering_advanced_top10.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSaved: /content/gaussian_filtering_advanced_top10.png")

## 10. Save Summary Report

In [ ]:
# Create summary report
summary = f"""
Gaussian Filtering Advanced - Analysis Summary
{'='*80}

Total Configurations Tested: {len(df)}

BEST CONFIGURATIONS:
{'-'*80}

Best Accuracy: {best_configs['accuracy']['accuracy']:.4f}
  Stage1: {best_configs['accuracy']['confidence_stage1']:.2f}
  Stage2: {best_configs['accuracy']['confidence_stage2']:.2f}

Best Precision: {best_configs['precision']['precision']:.4f}
  Stage1: {best_configs['precision']['confidence_stage1']:.2f}
  Stage2: {best_configs['precision']['confidence_stage2']:.2f}

Best Recall: {best_configs['recall']['recall']:.4f}
  Stage1: {best_configs['recall']['confidence_stage1']:.2f}
  Stage2: {best_configs['recall']['confidence_stage2']:.2f}

Best F1: {best_configs['f1']['f1']:.4f}
  Stage1: {best_configs['f1']['confidence_stage1']:.2f}
  Stage2: {best_configs['f1']['confidence_stage2']:.2f}
  Accuracy: {best_configs['f1']['accuracy']:.4f}
  Precision: {best_configs['f1']['precision']:.4f}
  Recall: {best_configs['f1']['recall']:.4f}

Best Specificity: {best_configs['specificity']['specificity']:.4f}
  Stage1: {best_configs['specificity']['confidence_stage1']:.2f}
  Stage2: {best_configs['specificity']['confidence_stage2']:.2f}

OVERALL STATISTICS:
{'-'*80}

Accuracy:     Mean={df['accuracy'].mean():.4f}, Std={df['accuracy'].std():.4f}
Precision:    Mean={df['precision'].mean():.4f}, Std={df['precision'].std():.4f}
Recall:       Mean={df['recall'].mean():.4f}, Std={df['recall'].std():.4f}
F1:           Mean={df['f1'].mean():.4f}, Std={df['f1'].std():.4f}
Specificity:  Mean={df['specificity'].mean():.4f}, Std={df['specificity'].std():.4f}
Pass Rate:    Mean={df['stage1_pass_rate'].mean():.4f}, Std={df['stage1_pass_rate'].std():.4f}

PCA Analysis:
{'-'*80}
PC1 Explained Variance: {pca.explained_variance_ratio_[0]:.2%}
PC2 Explained Variance: {pca.explained_variance_ratio_[1]:.2%}
Total Explained Variance: {pca.explained_variance_ratio_.sum():.2%}

Generated Visualizations:
{'-'*80}
1. gaussian_filtering_advanced_heatmaps.png
2. gaussian_filtering_advanced_metrics.png
3. gaussian_filtering_advanced_pr_curve.png
4. gaussian_filtering_advanced_pass_rate.png
5. gaussian_filtering_advanced_pca.png
6. gaussian_filtering_advanced_top10.png
"""

print(summary)

# Save to file
with open('/content/gaussian_filtering_advanced_summary.txt', 'w') as f:
    f.write(summary)

print("\nSummary saved to: /content/gaussian_filtering_advanced_summary.txt")

## Done!

**Analysis complete!**

**Generated files**:
- `gaussian_filtering_advanced_heatmaps.png`: Performance heatmaps
- `gaussian_filtering_advanced_metrics.png`: Metrics comparison
- `gaussian_filtering_advanced_pr_curve.png`: Precision-Recall curve
- `gaussian_filtering_advanced_pass_rate.png`: Stage1 pass rate analysis
- `gaussian_filtering_advanced_pca.png`: PCA visualization
- `gaussian_filtering_advanced_top10.png`: Top 10 configurations
- `gaussian_filtering_advanced_summary.txt`: Summary report

All files are saved in `/content/` directory